In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("cwd =", os.getcwd())


## 冷启动item统计

In [ ]:
from logging import getLogger
from typing import Union
import torch
import os
from accelerate import Accelerator
from torch.utils.data import DataLoader

from genrec.dataset import AbstractDataset
from genrec.model import AbstractModel
from genrec.tokenizer import AbstractTokenizer
from genrec.utils import get_config, init_seed, init_logger, init_device, \
    get_dataset, get_tokenizer, get_model, get_trainer, log
import argparse
from genrec.utils import parse_command_line_args, get_pipeline

category = 'Software'
if category == 'Video_Games':
		max_rows = 0.5
elif category == 'Office_Products':
		max_rows = 0.1
elif category == 'Cell_Phones_and_Accessories':
		max_rows = 0.1
elif category == 'Musical_Instruments': 
		max_rows = 0.5
elif category == 'Industrial_and_Scientific':
		max_rows = 0.5
elif category == 'Software':
		max_rows = 0.5
elif category == 'Baby_Products':
		max_rows = 0.2


number_per_item = 5
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model', type=str, default='TIGER', help='Model name')
    parser.add_argument('--dataset', type=str, default='AmazonReviews2023', help='Dataset name')
    parser.add_argument('--config_file', type=str, default="genrec/models/TIGER/config.yaml", help='Path to config file')
    return parser.parse_known_args()

args, unparsed_args = parse_args()
command_line_configs = {'pretrained_model_path': f'data/ckpt/TIGER_{category}/genrec_default_ori.pth',
												'category': category,
												'max_rows': max_rows
												}
# import ipdb; ipdb.set_trace()

model_name = args.model
config = get_config(
		model_name=args.model,
		dataset_name=args.dataset,
		config_dict=command_line_configs,
		config_file = args.config_file
)

# Accelerator
project_dir = os.path.join(
		config['tensorboard_log_dir'],
		config["dataset"],
		config["model"]
)
accelerator = Accelerator(log_with='tensorboard', project_dir= project_dir)
config['accelerator'] =  accelerator

# Seed and Logger
init_seed(config['rand_seed'],  config['reproducibility'])

raw_dataset = get_dataset(args.dataset)(config)
split_datasets = raw_dataset.split()   # 这里把数据集划分好了 返回的是字典 key是'train','val','test' value是Dataset对象



# 强冷启动统计


In [ ]:
# 假设 datasets 已经是你 return 的那个 dict，且 datasets['train']/['test'] 是 HF Dataset
train_ds = split_datasets["train"]
test_ds  = split_datasets["test"]

# 1) 训练集中出现过的所有 item（出现在任何位置都算“见过”）
train_items = set()
for seq in train_ds["item_seq"]:
    train_items.update(seq)

# 2) 测试集中每条序列的 target（默认取最后一个）
test_targets = [seq[-1] for seq in test_ds["item_seq"] if len(seq) > 0]

# 3) 统计 test target 从未在 train 出现过的条数
cold_cnt = sum(t not in train_items for t in test_targets)
total = len(test_targets)

print(f"test targets total: {total}")
print(f"cold targets (never in train): {cold_cnt}")
print(f"cold ratio: {cold_cnt/total:.4f}")

# （可选）看看有多少个不同的冷启动 item
cold_items = {t for t in test_targets if t not in train_items}
print(f"unique cold items in test targets: {len(cold_items)}")


# 弱冷启动统计



In [ ]:
from collections import Counter

train_ds = split_datasets["train"]
test_ds  = split_datasets["test"]

# 1) 统计训练集中每个 item 的出现次数（任何位置都算）
train_item_cnt = Counter()
for seq in train_ds["item_seq"]:
    train_item_cnt.update(seq)

# 2) 测试集每条序列的 target（最后一个）
test_targets = [seq[-1] for seq in test_ds["item_seq"] if len(seq) > 0]

# 3) 定义：训练集出现次数 < 5 的 test target 为“冷启动”
thr = 5
cold_cnt = sum(train_item_cnt.get(t, 0) < thr for t in test_targets)
total = len(test_targets)

print(f"test targets total: {total}")
print(f"cold targets (train freq < {thr}): {cold_cnt}")
print(f"cold ratio: {cold_cnt/total:.4f}")

# （可选）不同冷启动 item 的数量
cold_items = {t for t in test_targets if train_item_cnt.get(t, 0) < thr}
print(f"unique cold items in test targets: {len(cold_items)}")

# （可选）看一下这些冷启动 item 在 train 的频次分布
# 例如：0、1、2、3、4 各多少个（按“unique item”计）
freq_hist = Counter(train_item_cnt.get(t, 0) for t in cold_items)
print("cold item train freq histogram (by unique item):", dict(sorted(freq_hist.items())))


## Edit数据转化

In [ ]:
# test_dataset = split_datasets["cold_test"]
# len(test_dataset)


test_dataset = split_datasets["warm_test"]
len(test_dataset)

In [ ]:
# Load or encode sentence embeddings
import numpy as np
sent_emb_path = f"data/cache/{args.dataset}/{category}/processed/sentence-t5-base.sent_emb"
if os.path.exists(sent_emb_path):
		sent_embs = np.fromfile(sent_emb_path, dtype=np.float32).reshape(-1, config['sent_emb_dim'])

In [ ]:
sent_embs.shape

In [ ]:
item2id, id2item = raw_dataset.id_mapping['item2id'], raw_dataset.id_mapping['id2item']

In [ ]:
id2item, len(id2item)

In [ ]:
item2id, len(item2id)

In [ ]:
# PROMPT
# 
# 已知sent_embs是numpy：（18357, 768）
# split_datasets格式如下：{'train': Dataset({ 
#      features: ['user', 'item_seq'],
#      num_rows: 35598
#  }),
#  'val': Dataset({
#      features: ['user', 'item_seq'],
#      num_rows: 35598
#  }),
#  'test': Dataset({
#      features: ['user', 'item_seq'],
#      num_rows: 35598
#  }),
#  'warm_test': Dataset({
#      features: ['user', 'item_seq'],
#      num_rows: 35281
#  }),
#  'cold_test': Dataset({
#      features: ['user', 'item_seq'],
#      num_rows: 317
#  })}
# item2id:{'[PAD]': 0,
#  '1881509818': 1,
#  'B0048KGFHU': 2,
#  'B0081JJVUC': 3}
# id2item:['[PAD]', 
#  '1881509818',
#  'B0048KGFHU',
#  'B0081JJVUC',
#  'B000N8OIE8']
# 根据以下要求写python代码：
# 1 将cold_test的target item(即 item_seq的最后一个item) 设置为cold_item   split_datasets["cold_test"]是cold_test集，包含两个字段：user, item_seq格式如下： 
# split_datasets['cold_test'][0]数据样例如下：{'user': 'AIXZKN4ACSKI',
#  'item_seq': ['1881509818',
#   'B0048KGFHU',
#   'B0081JJVUC',
#   'B000N8OIE8',
#   'B004Y27DVY',
#   'B00D7ONGFC']}

# 2 利用相似度找到训练集中与1中target item的topk个最相似的item, 可以利用item2id找到item索引，在sent_embs中找到item的句向量表示, 最后用id2item 来映射成item，给出[{target item：[相似items]},...]。注意item2id的 id-1 才对应到 sent_embs 的下标；sent_embs 的下标的+1 才是id2item的索引。 

# 3 对于每个target item，找到训练集中与他们对应的相似items的位置，并将这些位置的item之前的item序列和target item构造为新的item_seq，将user和新的item_seq存入新的测试集合cold_test_augmented中。

           


In [ ]:

import numpy as np
from datasets import Dataset

def build_cold_or_warm_augmented(
	sent_embs: np.ndarray,
	split_datasets: dict,
	item2id: dict,
	id2item: list,
	topk: int = 20,
	max_aug_per_target: int | None = None,
	seed: int | None = None,   # 新增：控制随机性，None=不固定
	key_name: str = "cold_test"  # "cold" or "warm_test"
):
	rng = np.random.default_rng(seed)

	train_ds = split_datasets["train"]
	cold_ds = split_datasets[key_name]  # 不局限cold

	# ----------------------------
	# 0) 训练集中出现过的 item 集合
	# ----------------------------
	train_items = set()
	for seq in train_ds["item_seq"]:
		train_items.update(seq)
	train_items.discard("[PAD]")

	# 把训练 items 转成 sent_embs 的下标（id-1）
	train_emb_indices = []
	train_items_list = []
	for it in train_items:
		if it in item2id:
			iid = item2id[it]
			if iid > 0:
				emb_idx = iid - 1
				if 0 <= emb_idx < sent_embs.shape[0]:
					train_emb_indices.append(emb_idx)
					train_items_list.append(it)

	train_emb_indices = np.array(train_emb_indices, dtype=np.int64)
	train_embs = sent_embs[train_emb_indices]

	train_norm = np.linalg.norm(train_embs, axis=1, keepdims=True)
	train_norm = np.maximum(train_norm, 1e-12)
	train_embs_norm = train_embs / train_norm

	# -----------------------------------------
	# 1) cold_test target
	# -----------------------------------------
	cold_targets = []
	for seq in cold_ds["item_seq"]:
		if len(seq) > 0:
			cold_targets.append(seq[-1])

	seen = set()
	cold_targets_unique = []
	for t in cold_targets:
		if t not in seen:
			seen.add(t)
			cold_targets_unique.append(t)

	# -----------------------------------------
	# 2) target -> topk similar train items
	# -----------------------------------------
	target2sims = {}
	mapping_list = []

	def item_to_embidx(it: str) -> int | None:
		iid = item2id.get(it, None)
		if iid is None or iid <= 0:
			return None
		emb_idx = iid - 1
		if 0 <= emb_idx < sent_embs.shape[0]:
			return emb_idx
		return None


	count = 0
	for target in cold_targets_unique:
		t_idx = item_to_embidx(target)
		if t_idx is None:
			target2sims[target] = []
			mapping_list.append({target: []})
			continue

		t_vec = sent_embs[t_idx]
		t_norm = np.linalg.norm(t_vec)
		if t_norm < 1e-12:
			target2sims[target] = []
			mapping_list.append({target: []})
			continue

		t_vec_norm = t_vec / t_norm
		sims = train_embs_norm @ t_vec_norm  # (N_train_items,)

		sims_masked = sims
		items_masked = np.array(train_items_list, dtype=object)

		k = min(topk, sims_masked.shape[0])
		if k <= 0:
			target2sims[target] = []
			mapping_list.append({target: []})
			continue
		topk_idx = np.argpartition(-sims_masked, kth=k - 1)[:k]
		topk_idx = topk_idx[np.argsort(-sims_masked[topk_idx])] # 是在 tarin item list 中的idx
		# if t_idx in topk_idx:# t_idx 是在embeddding 中的idx
		# 	print("Found target in topk_idx, which should not happen.")
		sim_items = items_masked[topk_idx].tolist()
		target2sims[target] = sim_items #   target2sims[target] = [sim_item1, sim_item2,...]已经topk了
		if target in sim_items:
			# print("Warning: target item found in its own similar items.")
			count+=1
		mapping_list.append({target: sim_items})
	print(f"Total {count}/{len(cold_targets_unique)} targets found in their own similar items.")
	# -----------------------------------------
	# 3) 倒排：item -> [(user, seq, pos), ...]
	# -----------------------------------------
	inverted = {}
	train_users = train_ds["user"]
	train_seqs = train_ds["item_seq"]

	for user, seq in zip(train_users, train_seqs):
		for pos, it in enumerate(seq):
			inverted.setdefault(it, []).append((user, seq, pos))

	# -----------------------------------------
	# 4) 构造增强序列：先收集候选、去重，再随机采样
	# -----------------------------------------
	aug_users = []
	aug_item_seqs = []
	for target, sim_items in target2sims.items():
		if not sim_items:
			continue
		# (可选) 打乱 sim_items 顺序，避免总取相同的“头部相似item”
		sim_items_shuffled = sim_items.copy()
		rng.shuffle(sim_items_shuffled)

		# 候选池：用 set 去重（避免重复 new_seq）
		# 如果你还想避免同一个 user 重复，可以把 key 改成 (user, tuple(new_seq))
		candidate_set = set()
		candidates = []  # list of (user, new_seq)
		# for sim_it in sim_items_shuffled:
		# 	for user, seq, pos in inverted.get(sim_it, []):
		# 		prefix = seq[:pos]
		# 		new_seq = prefix + [target]
		# 		key = tuple(new_seq)  # 只按序列去重
		# 		if key not in candidate_set:
		# 			candidate_set.add(key)
		# 			candidates.append((user, new_seq))

		# 你可以在外面先设定想插几个 sim_it
		n_sim_insert = 2   # 比如随机插入 2 个 sim_it（你想插几个就改这里）
		for sim_it in sim_items_shuffled:
				for user, seq, pos in inverted.get(sim_it, []):
						prefix = list(seq[:pos])   # copy，避免影响原 seq
						# 1) 随机位置插入几个 sim_it
						for _ in range(n_sim_insert):
								ins_pos = int(rng.integers(0, len(prefix) + 1))
								prefix.insert(ins_pos, sim_it)

						# 2） 随机删除几个 prefix 中的 item
						n_prefix = len(prefix)
						n_del = min(2, n_prefix // 4)
						for _ in range(n_del):
								if len(prefix) == 0:
										break
								del_pos = int(rng.integers(0, len(prefix)))
								prefix.pop(del_pos)
						# 3) 再把 target 也随机插入一个位置（不是固定放末尾）
						
						ins_pos = int(rng.integers(0, len(prefix)/2 ))
						prefix.insert(ins_pos, target)

						# 4)  target 设置为序列最后一个位置
						new_seq = prefix + [target]
						key = tuple(new_seq)  # 只按序列去重
						if key not in candidate_set:
								candidate_set.add(key)
								candidates.append((user, new_seq))

		if not candidates:
			continue

		# # 随机采样最多 max_aug_per_target 条（无放回）
		# if max_aug_per_target is not None:
		# 	m = min(max_aug_per_target, len(candidates))
		# 	idx = rng.choice(len(candidates), size=m, replace=False)
		# 	chosen = [candidates[i] for i in idx]
		# else:
		# 	chosen = candidates
		if max_aug_per_target is not None:
			m = min(max_aug_per_target, len(candidates))
			# 先按长度降序；长度相同则打乱保持一定随机性
			rng.shuffle(candidates)  # 只影响 tie-break
			candidates_sorted = sorted(candidates, key=lambda x: len(x[1]), reverse=True)
			chosen = candidates_sorted[:m]
		else:
			chosen = candidates

		for user, new_seq in chosen:
			aug_users.append(user)
			aug_item_seqs.append(new_seq)
	print("key_name:", key_name)
	print("hhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhh")
	test_augmented = Dataset.from_dict({"user": aug_users, "item_seq": aug_item_seqs})
	if key_name == "cold_test":
		print(f"Cold test augmented dataset size: {len(test_augmented)}")
		split_datasets["cold_test_augmented"] = test_augmented
	elif key_name == "warm_test":
		print(f"Warm test augmented dataset size: {len(test_augmented)}")
		split_datasets["warm_test_augmented"] = test_augmented
	return mapping_list, test_augmented, split_datasets


In [ ]:
mapping_list, test_augmented, split_datasets = build_cold_or_warm_augmented(
    sent_embs=sent_embs,
    split_datasets=split_datasets,
    item2id=item2id,
    id2item=id2item,
    topk=10,
    max_aug_per_target=number_per_item,  # 可选：每个 target 最多生成 50 条，防止爆炸
		key_name="cold_test"  #cold_test warm_test
)

print(mapping_list[0])
print(test_augmented[0])
print(test_augmented.num_rows)
# replaced_items = list(set(replaced_items))
# print(replaced_items[:5],len(replaced_items))



In [ ]:
# # 获取所有相似items，去重，排除target items

# mapping_list[:5]
# sim_items = []
# for i in range(len(mapping_list)):
# 		values = list(mapping_list[i].values())[0]
# 		sim_items.extend(values)
# sim_items = list(set(sim_items))
# target_items = [list(m.keys())[0] for m in mapping_list]
# sim_wo_target = [x for x in sim_items if x not in set(target_items)]
# print(len(sim_wo_target))
# # 存储 sim_wo_target json
# import json
# sim_items_path = "data/Edit/cold_test_sim_items.json"
# with open(sim_items_path, "w") as f:
# 		json.dump(sim_wo_target, f)


In [ ]:
# # def build_cold_or_warm_augmented_simlar_items_as_history(
# import numpy as np
# from datasets import Dataset

# def build_cold_or_warm_augmented_simlar_items_as_history(
# 	sent_embs: np.ndarray,
# 	split_datasets: dict,
# 	item2id: dict,
# 	id2item: list,
# 	topk: int = 20,
# 	max_aug_per_target: int | None = None,
# 	seed: int | None = None,   # 新增：控制随机性，None=不固定
# 	key_name: str = "cold_test"  # "cold" or "warm_test"
# ):
# 	rng = np.random.default_rng(seed)

# 	train_ds = split_datasets["train"]
# 	cold_ds = split_datasets[key_name]  # 不局限cold

# 	# ----------------------------
# 	# 0) 训练集中出现过的 item 集合
# 	# ----------------------------
# 	train_items = set()
# 	for seq in train_ds["item_seq"]:
# 		train_items.update(seq)
# 	train_items.discard("[PAD]")

# 	# 把训练 items 转成 sent_embs 的下标（id-1）
# 	train_emb_indices = []
# 	train_items_list = []
# 	for it in train_items:
# 		if it in item2id:
# 			iid = item2id[it]
# 			if iid > 0:
# 				emb_idx = iid - 1
# 				if 0 <= emb_idx < sent_embs.shape[0]:
# 					train_emb_indices.append(emb_idx)
# 					train_items_list.append(it)

# 	train_emb_indices = np.array(train_emb_indices, dtype=np.int64)
# 	train_embs = sent_embs[train_emb_indices]

# 	train_norm = np.linalg.norm(train_embs, axis=1, keepdims=True)
# 	train_norm = np.maximum(train_norm, 1e-12)
# 	train_embs_norm = train_embs / train_norm

# 	# -----------------------------------------
# 	# 1) cold_test target
# 	# -----------------------------------------
# 	cold_targets = []
# 	target2users = {}  # 新增：target -> [users]，用测试集自身的 user

# 	for user, seq in zip(cold_ds["user"], cold_ds["item_seq"]):
# 			if len(seq) > 0:
# 					t = seq[-1]
# 					cold_targets.append(t)
# 					target2users.setdefault(t, []).append(user)
# 	seen = set()
# 	cold_targets_unique = []
# 	for t in cold_targets:
# 		if t not in seen:
# 			seen.add(t)
# 			cold_targets_unique.append(t)

# 	# -----------------------------------------
# 	# 2) target -> topk similar train items
# 	# -----------------------------------------
# 	target2sims = {}
# 	mapping_list = []

# 	def item_to_embidx(it: str) -> int | None:
# 		iid = item2id.get(it, None)
# 		if iid is None or iid <= 0:
# 			return None
# 		emb_idx = iid - 1
# 		if 0 <= emb_idx < sent_embs.shape[0]:
# 			return emb_idx
# 		return None


# 	count = 0
# 	for target in cold_targets_unique:
# 		t_idx = item_to_embidx(target)
# 		if t_idx is None:
# 			target2sims[target] = []
# 			mapping_list.append({target: []})
# 			continue

# 		t_vec = sent_embs[t_idx]
# 		t_norm = np.linalg.norm(t_vec)
# 		if t_norm < 1e-12:
# 			target2sims[target] = []
# 			mapping_list.append({target: []})
# 			continue

# 		t_vec_norm = t_vec / t_norm
# 		sims = train_embs_norm @ t_vec_norm  # (N_train_items,)

# 		sims_masked = sims
# 		items_masked = np.array(train_items_list, dtype=object)

# 		k = min(topk, sims_masked.shape[0])
# 		if k <= 0:
# 			target2sims[target] = []
# 			mapping_list.append({target: []})
# 			continue
# 		topk_idx = np.argpartition(-sims_masked, kth=k - 1)[:k]
# 		topk_idx = topk_idx[np.argsort(-sims_masked[topk_idx])] # 是在 tarin item list 中的idx
# 		# if t_idx in topk_idx:# t_idx 是在embeddding 中的idx
# 		# 	print("Found target in topk_idx, which should not happen.")
# 		# 随机生成的idx###################################################################需要改##########################################################
# 		# random_idx = rng.choice(len(train_items_list), size=k, replace=False)
# 		# topk_idx = random_idx
# 		# ##########################################################################################################################################
# 		sim_items = items_masked[topk_idx].tolist()
# 		target2sims[target] = sim_items
# 		if target in sim_items:
# 			# print("Warning: target item found in its own similar items.")
# 			count+=1
# 		mapping_list.append({target: sim_items})
# 	print(f"Total {count}/{len(cold_targets_unique)} targets found in their own similar items.")
	

	
# 	# -----------------------------------------
# 	# 4) 构造增强序列：先收集候选、去重，再随机采样
# 	# -----------------------------------------
# 	aug_users = []
# 	aug_item_seqs = []

# 	for target, sim_items in target2sims.items():
# 			if not sim_items:
# 					continue

# 			# 候选池：用 set 去重（避免重复 new_seq）
# 			candidate_set = set()
# 			candidates = []  # list of (user, new_seq)

# 			# 生成候选：如果 max_aug_per_target 给了，就生成多条（通过 shuffle 得到不同顺序）
# 			# 否则默认生成 1 条（按当前 sim_items 顺序）
# 			if max_aug_per_target is None:
# 					num_to_gen = 1
# 			else:
# 					num_to_gen = max_aug_per_target

# 			users_for_t = target2users.get(target, [])
# 			if not users_for_t:
# 					# 理论上不会发生（target 来自 cold_targets_unique），兜底
# 					users_for_t = [cold_ds["user"][0]]

# 			for i in range(num_to_gen):
# 					sim_items_shuffled = sim_items.copy()
# 					rng.shuffle(sim_items_shuffled)  # 通过打乱产生不同 history
# 					new_seq = sim_items_shuffled + [target]

# 					key = tuple(new_seq)
# 					if key in candidate_set:
# 							continue
# 					candidate_set.add(key)

# 					user = users_for_t[i % len(users_for_t)]  # 用测试集里真实 user，类型最稳
# 					candidates.append((user, new_seq))

# 			if not candidates:
# 					continue

# 			# 你原来的“按长度选更长”逻辑保留（这里长度基本都一样，等价于取前 m 条）
# 			if max_aug_per_target is not None:
# 					m = min(max_aug_per_target, len(candidates))
# 					rng.shuffle(candidates)  # tie-break
# 					candidates_sorted = sorted(candidates, key=lambda x: len(x[1]), reverse=True)
# 					chosen = candidates_sorted[:m]
# 			else:
# 					chosen = candidates

# 			for user, new_seq in chosen:
# 					aug_users.append(user)
# 					aug_item_seqs.append(new_seq)

# 	print("key_name:", key_name)
# 	test_augmented = Dataset.from_dict({"user": aug_users, "item_seq": aug_item_seqs})

# 	if key_name == "cold_test":
# 			print(f"Cold test augmented dataset size: {len(test_augmented)}")
# 			split_datasets["cold_test_augmented"] = test_augmented
# 	elif key_name == "warm_test":
# 			print(f"Warm test augmented dataset size: {len(test_augmented)}")
# 			split_datasets["warm_test_augmented"] = test_augmented

# 	return mapping_list, test_augmented, split_datasets

In [ ]:
# mapping_list, test_augmented, split_datasets = build_cold_or_warm_augmented_simlar_items_as_history(
#     sent_embs=sent_embs,
#     split_datasets=split_datasets,
#     item2id=item2id,
#     id2item=id2item,
#     topk=20,
#     max_aug_per_target=3,  # 可选：每个 target 最多生成 50 条，防止爆炸
# 		key_name="cold_test"  #cold_test
# )

# print(mapping_list[0])
# print(test_augmented[0])
# print(test_augmented.num_rows)


In [ ]:
# ## COV

# import json

# import numpy as np
# from datasets import Dataset

# def build_COV(
# 	split_datasets: dict,
# ):
# 	train_ds = split_datasets["train"]
# 	cold_ds = split_datasets["cold_test"]  # 不局限cold

# 	cold_targets = set()
# 	for i in range(len(cold_ds)):
# 		seq = cold_ds[i]["item_seq"]
# 		if len(seq) > 0:
# 			cold_targets.add(seq[-1])
# 	# ----------------------------
# 	# 筛选出训练集的target 不在cold_targets中的数据
# 	filtered_train_users = []
# 	filtered_train_seqs = []
# 	for user, seq in zip(train_ds["user"], train_ds["item_seq"]):
# 		if len(seq) == 0:
# 			continue
# 		if seq and seq[-1] not in cold_targets:
# 			filtered_train_users.append(user)
# 			filtered_train_seqs.append(seq)
	
# 	test_augmented = Dataset.from_dict({"user": filtered_train_users, "item_seq": filtered_train_seqs})
# 	print("original train dataset size:", len(train_ds))
# 	print(f"COV dataset size: {len(test_augmented)}")
# 	split_datasets["COV"] = test_augmented

# 	return test_augmented, split_datasets

	



In [ ]:
# test_augmented, split_datasets = build_COV(
#     split_datasets=split_datasets,
# )
# print(test_augmented[0])
# print(test_augmented.num_rows)
# # replaced_items = list(set(replaced_items))
# # print(replaced_items[:5],len(replaced_items))



In [ ]:
split_datasets.keys()


In [ ]:
split_datasets["cold_test_augmented"][0]

## Tokenize

In [ ]:
# Tokenize 两个数据集
tokenizer = get_tokenizer(model_name)(config, raw_dataset)
tokenized_datasets =  tokenizer.tokenize({"cold_test_augmented":split_datasets["cold_test_augmented"]})  ## 这里把数据集都tokenize了（变成了sids）,"cold_test":split_datasets["cold_test"]
tokenized_datasets

# # # COV
# tokenizer = get_tokenizer(model_name)(config, raw_dataset)
# tokenized_datasets =  tokenizer.tokenize({"COV":split_datasets["COV"],"cold_test_augmented":split_datasets["cold_test_augmented"],"cold_test":split_datasets["cold_test"]})  ## 这里把数据集都tokenize了（变成了sids）
# tokenized_datasets

# # Tokenize cold_test only
# tokenizer = get_tokenizer(model_name)(config, raw_dataset)
# tokenized_datasets =  tokenizer.tokenize({"cold_test":split_datasets["cold_test"]})  ## 这里把数据集都tokenize了（变成了sids）
# tokenized_datasets

# # Tokenize cold_test only
# tokenizer = get_tokenizer(model_name)(config, raw_dataset)
# tokenized_datasets =  tokenizer.tokenize({"warm_test_augmented":split_datasets["warm_test_augmented"]})  ## 这里把数据集都tokenize了（变成了sids）
# tokenized_datasets
# # Model
# with accelerator.main_process_first():
# 		model = get_model(model_name)(config,  raw_dataset,  tokenizer)


In [ ]:

out_path=f"data/Edit/{category}/edit_requests_cold_test_augmented_{number_per_item}_train.json"

tokenized_datasets["cold_test_augmented"].save_to_disk(out_path)

In [ ]:
# from datasets import load_from_disk

# out_path=f"data/Edit/Video_Games/edit_requests_cold_test_augmented_5_train.json"

# datasets = load_from_disk(out_path)
# datasets

In [ ]:
# tokenized_datasets["cold_test"][0]

In [ ]:
# tokenized_datasets['cold_test_augmented']["input_ids"][0].shape

In [ ]:
# 已知 tokenized_datasets['cold_test_augmented'][0] 数据样例如下：{'input_ids': tensor([1025,   17,  378,  568,  770,    1,  472,  573,  770, 1026,    0,    0,
#             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
#             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
#             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
#             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
#             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
#             0,    0,    0,    0,    0,    0,    0,    0,    0,    0]),
#  'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
#          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
#          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
#          0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
#  'labels': tensor([  68,  413,  598,  770, 1026])}
# 请构造成json文件格式如下：            
# edit_requests：[{
#                 "history": input_ids,#( list[int])
#                 "target_sids": labels,#( list[str])
#                 "case_id": str(len(edit_requests))
#             },...]

In [ ]:
import json
import torch

def build_edit_requests_json(tokenized_datasets, split="cold_test_augmented"):
    ds = tokenized_datasets[split]
    edit_requests = []

    for i in range(len(ds)):
        ex = ds[i]

        input_ids = ex["input_ids"]
        attn_mask = ex.get("attention_mask", None)
        labels = ex["labels"]

        # 1) input_ids -> history (list[int])
        if isinstance(input_ids, torch.Tensor):
            input_ids = input_ids.tolist()

        # 可选：用 attention_mask 去掉 padding（推荐）
        # if attn_mask is not None:
        #     if isinstance(attn_mask, torch.Tensor):
        #         attn_mask = attn_mask.tolist()
        #     valid_len = int(sum(attn_mask))
        #     history = input_ids[:valid_len]
        # else:
        #     history = input_ids

        # 如果你想保留 padding（不截断），改成：
        history = input_ids

        # 2) labels -> target_sids (list[str])
        if isinstance(labels, torch.Tensor):
            labels = labels.tolist()
        target_sids = [str(x) for x in labels]

        edit_requests.append({
            "history": history,
            "target_sids": target_sids,
            "case_id": str(len(edit_requests))
        })

    # payload = {"edit_requests": edit_requests}
    return edit_requests

# # 用法：
# payload = build_edit_requests_json(tokenized_datasets, "cold_test_augmented")
# print(payload[0])

# out_path="data/Edit/edit_requests.json"
# with open(out_path, "w", encoding="utf-8") as f:
# 		json.dump(payload, f, ensure_ascii=False, indent=2)


In [ ]:
tokenized_datasets.keys()

In [ ]:
# 用法：
payload = build_edit_requests_json(tokenized_datasets, "cold_test_augmented")
print(payload[0])

out_path=f"data/Edit/{category}/edit_requests_cold_test_augmented_{number_per_item}.json"

os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
		json.dump(payload, f, ensure_ascii=False, indent=2)
print(len(payload))

In [ ]:
# 用法：
payload = build_edit_requests_json(tokenized_datasets, "COV")
print(payload[0])

out_path=f"data/Edit/{category}/edit_requests_COV.json"

os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
		json.dump(payload, f, ensure_ascii=False, indent=2)
print(len(payload))

In [ ]:
# import json 

# with open(f"data/Edit/{category}/edit_requests_cold_test_augmented.json", "r", encoding="utf-8") as f:
# 		data = json.load(f)
# print(len(data))

In [ ]:
# payload = build_edit_requests_json(tokenized_datasets, "cold_test")
# print(payload[0])

# out_path=f"data/Edit/{category}/edit_requests_gt.json"
# with open(out_path, "w", encoding="utf-8") as f:
# 		json.dump(payload, f, ensure_ascii=False, indent=2)


In [ ]:
# import json 

# with open("data/Edit/edit_requests_similar_as_history.json", "r", encoding="utf-8") as f:
# 		data = json.load(f)
# print(len(data))

In [ ]:

# print([target["history"] for target in payload[0:2]])